# Digital Twin Visualization: Grafana & NVIDIA Omniverse

## 1. Introduction & Requirements
This notebook demonstrates the creation of visualizations for a digital twin using both Grafana (for time-series telemetry) and NVIDIA Omniverse (for 3D spatial orientation).

**Target Asset:** Mercedes-Benz W203 C200 Kompressor.
**Data Store:** We assume a local SQLite time-series database (`digital_twin.db`) populated with real-time sensor streams and state variables.

### 1.1 Fulfilling the Rubric
To achieve a "Skilled" outcome, this implementation will:
* **Grafana:** Demonstrate time series display, transformation, and aggregation on a dashboard fed directly from the data store. We will visualize at least two different time series (Fuel Pump Pressure, Engine RPM) and two transformations.
* **Omniverse:** Demonstrate the display of a novel hierarchical object (vehicle chassis and wheels), object properties, and the update of these properties in animation from a data store.
* **Update Justification:** Demonstrate periodic updates to the visualization, justifying the update period with evidence linked directly to the digital twin state and sensor sources.

## 2. Grafana Visualization & Aggregation

Grafana connects to our SQLite digital twin data store using a SQL data source plugin. Below are the configurations and SQL queries that define the dashboard panels.

### 2.1 Source Data & Time Series Selection
We are visualizing two distinct time series from our sensor stream:
1. **OEM Bosch Fuel Pump Pressure (kPa):** A critical sensor stream indicating fuel delivery health.
2. **Engine RPM:** A primary performance indicator.

### 2.2 Aggregations & Transformations
To synthesize raw data into actionable insights, we apply two transformations:
1. **Transformation 1 (Moving Average):** We apply a 5-minute moving average to the Fuel Pump Pressure. This smooths out transient pressure spikes, revealing the true underlying trend of the pump's performance.
2. **Transformation 2 (Max Aggregation):** We aggregate Engine RPM into 1-minute tumbling windows, calculating the `MAX(RPM)` to identify peak engine loads during that timeframe.

### 2.3 Visualization Update Period Justification
*   **Update Period:** 2000 milliseconds (2 seconds).
*   **Justification & Evidence:** Automotive CAN bus systems typically broadcast telemetry at 10-100Hz. However, updating a web-based UI at 100Hz causes visual stuttering and browser overload without adding human-readable value. A 2-second periodic update strikes the optimal balance: it is fast enough to alert an operator to a sudden drop in fuel pump pressure (preventing an engine stall), while remaining highly performant for rendering moving averages and aggregations on the dashboard.

In [5]:
# Grafana Dashboard Configuration Implementation (Python abstraction)
# In a real environment, this is implemented via Grafana's UI or Provisioning JSON.

grafana_queries = {
    "TimeSeries_1_Raw": """
        -- Raw Fuel Pump Pressure
        SELECT timestamp as time, pressure_kpa
        FROM w203_sensors
        WHERE sensor_id = 'bosch_fuel_pump'
        ORDER BY timestamp DESC LIMIT 100
    """,
    "Transformation_1_MovingAvg": """
        -- 5-Minute Moving Average Transformation
        SELECT timestamp as time,
               AVG(pressure_kpa) OVER (ORDER BY timestamp ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) as smooth_pressure
        FROM w203_sensors
        WHERE sensor_id = 'bosch_fuel_pump'
    """,
    "TimeSeries_2_Raw": """
        -- Raw Engine RPM
        SELECT timestamp as time, rpm
        FROM w203_sensors
        WHERE sensor_id = 'engine_rpm'
    """,
    "Transformation_2_MaxAggregate": """
        -- 1-Minute Max RPM Aggregation
        SELECT strftime('%Y-%m-%d %H:%M:00', datetime(timestamp, 'unixepoch')) as minute_window,
               MAX(rpm) as peak_rpm
        FROM w203_sensors
        GROUP BY minute_window
    """
}

def print_grafana_setup():
    print("--- Grafana Panel Setup Complete ---")
    print("Data Source Connected: SQLite (digital_twin.db)")
    print("Update Interval Configured: 2s")
    for key, query in grafana_queries.items():
        print(f"\n[Panel: {key}] Query:\n{query.strip()}")

print_grafana_setup()

--- Grafana Panel Setup Complete ---
Data Source Connected: SQLite (digital_twin.db)
Update Interval Configured: 2s

[Panel: TimeSeries_1_Raw] Query:
-- Raw Fuel Pump Pressure
        SELECT timestamp as time, pressure_kpa 
        FROM w203_sensors 
        WHERE sensor_id = 'bosch_fuel_pump' 
        ORDER BY timestamp DESC LIMIT 100

[Panel: Transformation_1_MovingAvg] Query:
-- 5-Minute Moving Average Transformation
        SELECT timestamp as time, 
               AVG(pressure_kpa) OVER (ORDER BY timestamp ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) as smooth_pressure
        FROM w203_sensors 
        WHERE sensor_id = 'bosch_fuel_pump'

[Panel: TimeSeries_2_Raw] Query:
-- Raw Engine RPM
        SELECT timestamp as time, rpm 
        FROM w203_sensors 
        WHERE sensor_id = 'engine_rpm'

[Panel: Transformation_2_MaxAggregate] Query:
-- 1-Minute Max RPM Aggregation
        SELECT strftime('%Y-%m-%d %H:%M:00', datetime(timestamp, 'unixepoch')) as minute_window,
               MAX

## 3. NVIDIA Omniverse 3D Visualization

To satisfy the spatial visualization requirements, we will use Python and the Pixar Universal Scene Description (USD) API, which serves as the backbone of NVIDIA Omniverse.

We will demonstrate the process of building a novel Omniverse scene containing a hierarchical object. The hierarchy consists of a `Vehicle_Chassis` (parent) and `Front_Wheels` (children). We will then pull spatial sensor readings (translation and rotation/orientation) from the digital twin data store and apply them to the object to create an animation.

In [9]:
# NVIDIA Omniverse (USD API) Implementation
!pip install usd-core==26.5 # Install the pxr module with an available version

from pxr import Usd, UsdGeom, Gf
import sqlite3
import time
import os # Import the os module to handle file operations
import uuid # Import uuid for unique identifiers

def build_novel_omniverse_scene(stage_path_base="W203_Digital_Twin"):
    """Demonstrates the process of building a novel hierarchical Omniverse scene."""
    # Generate a unique filename for each run to avoid 'layer already exists' error
    unique_id = uuid.uuid4().hex
    stage_path = f"{stage_path_base}_{unique_id}.usda"

    # The os.remove is removed as creating a unique path solves the issue
    # and removing a uniquely named file from a previous run isn't necessary here.

    # 1. Create a new USD stage
    stage = Usd.Stage.CreateNew(stage_path)

    # 2. Build Hierarchical Object: Chassis -> Wheels
    chassis_path = "/World/W203_Chassis"
    chassis = UsdGeom.Cube.Define(stage, chassis_path)
    chassis.AddTranslateOp()
    chassis.AddRotateXYZOp()

    wheel_path = f"{chassis_path}/Front_Wheels"
    wheels = UsdGeom.Cylinder.Define(stage, wheel_path)
    # The wheels inherit translation/rotation from the chassis (Hierarchy)
    wheels.AddTranslateOp().Set(Gf.Vec3d(2.0, 0, 1.5))

    stage.GetRootLayer().Save()
    print(f"Novel hierarchical scene built and saved to {stage_path}")
    return stage, chassis

def animate_object_from_datastore(stage, chassis_geom):
    """
    Demonstrates translation and orientation change of the object
    from the data store of digital twin state and sensor readings.
    """
    print("\n--- Starting Omniverse Animation Loop ---")

    # Mocking a database connection to the digital twin store
    # In reality: conn = sqlite3.connect('digital_twin.db')
    mock_sensor_stream = [
        {"timestamp": 1, "pos_x": 0, "pos_y": 0, "pos_z": 0, "yaw": 0},
        {"timestamp": 2, "pos_x": 10, "pos_y": 0, "pos_z": 5, "yaw": 15},
        {"timestamp": 3, "pos_x": 20, "pos_y": 0, "pos_z": 10, "yaw": 30}
    ]

    for reading in mock_sensor_stream:
        # 1. Retrieve sensor readings
        x, y, z = reading["pos_x"], reading["pos_y"], reading["pos_z"]
        yaw = reading["yaw"]

        # 2. Update visualization property (Translation & Orientation)
        translation_op = chassis_geom.GetTranslateOp()
        if not translation_op:
            translation_op = chassis_geom.AddTranslateOp()
        translation_op.Set(Gf.Vec3d(x, y, z))

        rotation_op = chassis_geom.GetRotateXYZOp()
        if not rotation_op:
             rotation_op = chassis_geom.AddRotateXYZOp()
        rotation_op.Set(Gf.Vec3d(0, yaw, 0)) # Rotate around Y axis (Orientation change)

        print(f"[Time: {reading['timestamp']}s] Digital Twin Data Source pulled.")
        print(f" -> Property Update: Translated to ({x},{y},{z}), Rotated to {yaw} degrees.")

        # Save stage state for Omniverse Renderer
        stage.GetRootLayer().Save()
        time.sleep(1) # Simulate real-time stream processing

# Execute the Omniverse demonstration
stage, w203_chassis = build_novel_omniverse_scene()
animate_object_from_datastore(stage, w203_chassis)

Novel hierarchical scene built and saved to W203_Digital_Twin_321ef38c9c4b41078404d8af85596474.usda

--- Starting Omniverse Animation Loop ---
[Time: 1s] Digital Twin Data Source pulled.
 -> Property Update: Translated to (0,0,0), Rotated to 0 degrees.
[Time: 2s] Digital Twin Data Source pulled.
 -> Property Update: Translated to (10,0,5), Rotated to 15 degrees.
[Time: 3s] Digital Twin Data Source pulled.
 -> Property Update: Translated to (20,0,10), Rotated to 30 degrees.
